In [1]:
from dataloader import *
from augmentations import *
from models import *
from training import *
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from statistics import mean, stdev
import copy

## Models

In [ ]:
models = [#"SmallCNN", 
          "DenseNet121", 
          "VGG16_BN", 
          "ResNet", ]

## Experiment loop for hyperparameter selection

In [11]:
params = {
     #training params
    "batch_size": [8, 32, 128],
    "learning_rate": [1e-3, 1e-4, 1e-5],
    "optimizer": ['Adam', 'SGD'],
    #regularization params
    "dropout": [0.2, 0.5],
    "weight_decay": [0, 1e-3, 0.5]
}

# Create experiment grid
experiment_grid = [dict(zip(params.keys(), v)) for v in product(*params.values())]

In [ ]:
def run_experiment(model_name, experiment_config, epoch_number=2, datasize=9000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    current_dropout = experiment_config.get("dropout", 0.2)
    model = get_model(model_name, dropout_rate=current_dropout)   
    model.to(device)

    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data
    transformations = basic_transforms(augmentation_type=None, model_type=model_name)
    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    train_dataset = get_subset(train_dataset, datasize)
    val_dataset = get_subset(val_dataset, datasize)
    test_dataset = get_subset(test_dataset, datasize)
    
    train_loader = get_train_dataloaders(train_dataset, batch_size=experiment_config["batch_size"])
    val_loader, test_loader = get_val_test_dataloaders(val_dataset, test_dataset, batch_size=experiment_config["batch_size"])

    criterion = torch.nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_model_weights = None
    best_val_metrics = {}

    for epoch in range(epoch_number):
        loss_score = train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device, criterion)

        print(f"Epoch number: {epoch}; training loss: {loss_score:.5f}; val loss: {scores['loss']:.5f}")

        # Zapisujemy wagi modelu TYLKO wtedy, gdy poprawił się wynik na walidacji
        if scores['loss'] < best_val_loss:
            best_val_loss = scores['loss']
            best_model_weights = copy.deepcopy(model.state_dict())
            
            # Zapisujemy metryki z tej najlepszej epoki z prefixem "val_"
            best_val_metrics = {f"val_{k}": v for k, v in scores.items()}

    # PRZED TESTOWANIEM: Wczytujemy najlepsze wagi z całego treningu
    print("Loading best model weights for testing...")
    model.load_state_dict(best_model_weights)

    # Testujemy NAJLEPSZY model
    raw_test_scores = validate(model, test_loader, device, criterion)
    
    # Dodajemy prefix "test_" do wyników testowych dla porządku
    test_scores = {f"test_{k}": v for k, v in raw_test_scores.items()}

    # Zwracamy połączone słowniki z najlepszej epoki walidacji oraz z testu
    return {**best_val_metrics, **test_scores}

## Sprawdzenie na mniejszej liczbie epok i mniejszym zbiorze (10%)

In [ ]:
results = []

for model in models:
    for i, config in enumerate(experiment_grid):

        #if i ==3:
        #    break

        print(f"Configuration number {i+1}, model: {model} config params: {config}")

        if model == "SmallCNN":
            epoch_number = 50
        elif model == "ResNet":
            epoch_number = 20
        else:
            break

        score = run_experiment(model, config, epoch_number=epoch_number, datasize=900)
        d = {
            "model": model,
            "config": config}
        results.append({
            **d, **score
        })

#df = pd.DataFrame(results)
#df.to_csv("experiment_results_23.02.2026.csv", index=False)

Configuration number 1, model: ResNet config params: {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0}
Training Loop


  8%|▊         | 869/11250 [00:16<03:16, 52.88it/s]


KeyboardInterrupt: 

In [10]:
# save configurations for 10 models with best accuracy on validation set from a csv file with results 
import ast

experiment_grid_small = {}
for model in models:
    results_df = pd.read_csv("experiment_results_23.02.2026.csv")
    results_df = results_df[results_df["model"] == model]
    results_df = results_df.sort_values(by="mean_acc", ascending=False).head(5)
    #change config from string to dict
    results_df["config"] = results_df["config"].apply(lambda x: ast.literal_eval(x))
    experiment_grid_small[model] = results_df["config"].tolist()
    print(f"Model: {model}, best configs: {experiment_grid_small[model]}, best acc: {results_df['mean_acc'].tolist()}")
    

Model: ResNet, best configs: [{'batch_size': 32, 'learning_rate': 0.001, 'optimizer': 'SGD', 'dropout': 0.5, 'weight_decay': 0.001}, {'batch_size': 32, 'learning_rate': 0.001, 'optimizer': 'SGD', 'dropout': 0.5, 'weight_decay': 0}, {'batch_size': 32, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}, {'batch_size': 32, 'learning_rate': 0.001, 'optimizer': 'SGD', 'dropout': 0.2, 'weight_decay': 0}, {'batch_size': 32, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.5, 'weight_decay': 0}], best acc: [0.7209222222222222, 0.7204111111111111, 0.7203777777777778, 0.7189888888888889, 0.7189333333333333]
Model: SmallCNN, best configs: [{'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'SGD', 'dropout': 0.5, 'weight_decay': 0}, {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.5, 'weight_decay': 0}, {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'SGD', 'dropout': 0.2, 'weight_decay': 0.001}, {'batch_size': 8, 'lea

## Training on full data and more epochs but considering top5 configs from previous experiment

In [ ]:
results = []

for model in models:
    for i, config in enumerate(experiment_grid_small[model]):
        print(f"Configuration number {i+1}, model: {model} config params: {config}")

        if model == "SmallCNN":
            epoch_number = 100
        elif model == "ResNet":
            epoch_number = 20
        else:
            break
        
        score = run_experiment(model, config, epoch_number=epoch_number, datasize=9000)
        d = {
            "model": model,
            "config": config}
        results.append({
            **d, **score
        })

df = pd.DataFrame(results)
df.to_csv("experiment_results_24.02.2026.csv", index=False)

## Training 2 other models 

In [5]:
results = []

for model in models:
    #take only 1 configuration for each model 
    for i, config in enumerate(experiment_grid): 
        print(f"Configuration number {i+1}, model: {model} config params: {config}")

        if model == "DenseNet121" or model == "VGG16_BN":
            epoch_number = 50
        else:
            break
        
        score = run_experiment(model, config, epoch_number=epoch_number, datasize=450)
        d = {
            "model": model,
            "config": config}
        results.append({
            **d, **score
        })

df = pd.DataFrame(results)
df.to_csv("experiment_results_25.02.2026.csv", index=False)

Configuration number 1, model: ResNet config params: {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0}
Configuration number 1, model: SmallCNN config params: {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0}
Configuration number 1, model: DenseNet121 config params: {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0}


/home/igor-lechoszest/anaconda3/envs/AI_Diag/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/igor-lechoszest/anaconda3/envs/AI_Diag/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Training Loop


100%|██████████| 563/563 [00:05<00:00, 95.01it/s] 


Validation Loop


...

Configuration number 2, model: DenseNet121 config params: {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}
Training Loop


100%|██████████| 563/563 [00:05<00:00, 104.10it/s]


Validation Loop


 58%|█████▊    | 324/563 [00:04<00:03, 76.43it/s]


KeyboardInterrupt: 

In [6]:
results

[{'model': 'DenseNet121',
  'config': {'batch_size': 8,
   'learning_rate': 0.001,
   'optimizer': 'Adam',
   'dropout': 0.2,
   'weight_decay': 0},
  'val_loss': 1.7640879518397112,
  'val_accuracy': 0.372,
  'val_precision': 0.435034896597365,
  'val_recall': 0.37200000000000005,
  'val_f1': 0.3265139964909027,
  'test_loss': 1.7526084738345171,
  'test_accuracy': 0.38666666666666666,
  'test_precision': 0.44255191750230916,
  'test_recall': 0.3866666666666667,
  'test_f1': 0.3366053695441253}]

## Model Ranking

In [1]:
for model in models:
    model_results = list(filter(lambda res: res["model"] == model, results))
    sorted_results = sorted(model_results, key=lambda x: -x["accuracy"])
    print(f"Ranking for model {model}")
    for i,res in enumerate(sorted_results):
        print("Rank number: ",i, end=" ")
        print(sorted_results[i])
    print("="*50)

NameError: name 'models' is not defined

### Tutaj wybieramy najlepsze konfiguracje (jedna na kazdy model) i zaczynamy testowanie augmentacji

## Experiment loop for auqmentation techniques

In [3]:
best_conf_resnet = {
    'batch_size': 32, 
    'learning_rate': 0.01,   
    'optimizer': 'SGD',      
    'dropout': 0.2, 
    'weight_decay': 0.0001   
}

best_conf_vgg16 = {
    'batch_size': 32, 
    'learning_rate': 0.01, 
    'optimizer': 'SGD', 
    'dropout': 0.5,          
    'weight_decay': 0.0001
}

best_conf_densenet = {
    'batch_size': 32, 
    'learning_rate': 0.001,  
    'optimizer': 'Adam', 
    'dropout': 0.2, 
    'weight_decay': 0.0001
}

best_conf_smallcnn = {
    'batch_size': 256,      
    'learning_rate': 0.001, 
    'optimizer': 'Adam', 
    'dropout': 0.1, 
    'weight_decay': 0.0001
}

best_confs = {
    "ResNet": best_conf_resnet,
    "DenseNet": best_conf_densenet, 
    "VGG16_BN": best_conf_vgg16,
    "SmallCNN": best_conf_smallcnn
}

In [4]:
# standard operations
basic_augmentations = ["flip", "shift", "rotation", None]
# more advanced data augmentation
advanced_augmentations = 'cutmix' 

In [5]:

def run_experiment_augmentation(model_name, experiment_config, augmentation=None, cutmix_collate_fn=None, epoch_number=2, datasize=900):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    current_dropout = experiment_config.get("dropout", 0.2)
    model = get_model(model_name, dropout_rate=current_dropout)   
    model.to(device)

    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data
    transformations = basic_transforms(augmentation_type=augmentation, model_type=model_name)

    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    # dynamiczny rozmiar podzbiorów oparty na argumencie datasize
    train_dataset = get_subset(train_dataset, datasize)
    val_dataset = get_subset(val_dataset, datasize)
    test_dataset = get_subset(test_dataset, datasize)

    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=cutmix_collate_fn,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_model_weights = None
    best_val_metrics = {}

    for epoch in range(epoch_number):

        loss_score = train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device, criterion)

        print(f"Epoch number: {epoch}; training loss: {loss_score:.5f}; val loss: {scores['loss']:.5f}")

        # Zapisujemy najlepsze wagi i metryki
        if scores['loss'] < best_val_loss:
            best_val_loss = scores['loss']
            best_model_weights = copy.deepcopy(model.state_dict())
            
            # Formatujemy nazwy metryk dla zbioru walidacyjnego
            best_val_metrics = {f"val_{k}": v for k, v in scores.items()}

    # PRZED TESTOWANIEM: Wczytujemy wagi z najlepszej epoki
    print("Loading best model weights for testing...")
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)

    raw_test_scores = validate(model, test_loader, device, criterion)
    
    # Formatujemy nazwy metryk dla zbioru testowego
    test_scores = {f"test_{k}": v for k, v in raw_test_scores.items()}

    return {**best_val_metrics, **test_scores}

In [6]:

results_augm = []

for model_name in models:
    if model_name == "SmallCNN":
        current_epochs = 50
    else:
        current_epochs = 10

    for augm in basic_augmentations:

        # Ładniejsze wypisywanie dla baseline'u
        augm_label = augm if augm is not None else "baseline (brak)"
        print(f"\n{'='*50}")
        print(f"Rozpoczynam eksperyment: Model: {model_name} | Augmentation: {augm_label}")
        print(f"{'='*50}")

        # Wywołanie z jawnym podaniem epok i całego zbioru (datasize=9000)
        score = run_experiment_augmentation(
            model_name=model_name,
            experiment_config=best_confs[model_name], 
            augmentation=augm, 
            cutmix_collate_fn=None,
            epoch_number=current_epochs,        
            datasize=9000          # 9000 * 10 klas = 90 000 (cały zbiór CINIC-10)
        )
        
        d = {
            "model": model_name,
            "augmentation": augm_label
        }
        
        results_augm.append({**d, **score})

    # ==========================================
    # Oddzielny blok dla augmentacji CutMix
    # ==========================================
    print(f"\n{'='*50}")
    print(f"Rozpoczynam eksperyment: Model: {model_name} | Augmentation: cutmix")
    print(f"{'='*50}")
    
    score = run_experiment_augmentation(
        model_name=model_name,
        experiment_config=best_confs[model_name], 
        augmentation=None, # Przy cutmix nie nakładamy bazowych transformacji
        cutmix_collate_fn=cutmix_collate_fn,
        epoch_number=current_epochs,
        datasize=9000
    )
    
    d = {
        "model": model_name,
        "augmentation": 'cutmix'
    }
        
    results_augm.append({**d, **score})

# Po zakończeniu wszystkich pętli konwertujemy listę słowników na DataFrame
# Dzięki temu będziesz mógł to łatwo wyświetlić i zapisać!
df_results = pd.DataFrame(results_augm)
print("\nGotowe! Wyniki eksperymentów:")
print(df_results)
df_results.to_csv("wyniki_augmentacji_27_03_2026.csv", index=False) # Odkomentuj, żeby zapisać do pliku


Rozpoczynam eksperyment: Model: SmallCNN | Augmentation: flip
Training Loop


100%|██████████| 352/352 [00:17<00:00, 20.12it/s]


Validation Loop


100%|██████████| 352/352 [00:11<00:00, 29.40it/s]


Epoch number: 0; training loss: 2.29123; val loss: 2.26762
Training Loop


...

100%|██████████| 352/352 [00:11<00:00, 30.18it/s]


Epoch number: 49; training loss: 1.81951; val loss: 1.39202
Loading best model weights for testing...
Validation Loop


100%|██████████| 352/352 [00:11<00:00, 30.25it/s]



Rozpoczynam eksperyment: Model: DenseNet121 | Augmentation: flip


KeyError: 'DenseNet121'

In [8]:
df_results = pd.DataFrame(results_augm)
print("\nGotowe! Wyniki eksperymentów:")
print(df_results)
df_results.to_csv("wyniki_augmentacji_SmallCNN_27_03_2026.csv", index=False) # Odkomentuj, żeby zapisać do pliku


Gotowe! Wyniki eksperymentów:
      model     augmentation  val_loss  val_accuracy  val_precision  \
0  SmallCNN             flip  1.265189      0.540322       0.539875   
1  SmallCNN            shift  1.293216      0.525800       0.525651   
2  SmallCNN         rotation  1.165214      0.580900       0.581431   
3  SmallCNN  baseline (brak)  1.134895      0.592356       0.589724   
4  SmallCNN           cutmix  1.392023      0.518533       0.509430   

   val_recall    val_f1  test_loss  test_accuracy  test_precision  \
0    0.540322  0.537832   1.272354       0.538756        0.538487   
1    0.525800  0.516335   1.299712       0.524133        0.524861   
2    0.580900  0.580378   1.171723       0.578333        0.578873   
3    0.592356  0.590510   1.143293       0.589822        0.587040   
4    0.518533  0.511784   1.395607       0.518056        0.508981   

   test_recall   test_f1  
0     0.538756  0.536372  
1     0.524133  0.514909  
2     0.578333  0.577923  
3     0.589822  0.5

## Model Ranking

In [ ]:
for model in models:
    model_results = list(filter(lambda res: res["model"] == model, results_augm))
    sorted_results = sorted(model_results, key=lambda x: -x["accuracy"])
    print(f"Ranking for model {model}")
    for i,res in enumerate(sorted_results):
        print("Rank number: ",i, end=" ")
        print(sorted_results[i])
    print("="*50)

Ranking for model ResNet
Rank number:  0 {'model': 'ResNet', 'augmentation': 'shift', 'mean_acc': 0.2833333333333333, 'std_acc': 0.07071067811865474, 'mean_f1': 0.19761650886330612, 'std_f1': 0.04490380308558294, 'mean_recall': 0.2555672268907563, 'std_recall': 0.036068387914305354, 'mean_precision': 0.2562156862745098, 'std_precision': 0.12687714028702146, 'loss': 2.2505622704823813, 'accuracy': 0.25555555555555554, 'precision': 0.2663059163059163, 'recall': 0.31682539682539684, 'f1': 0.21395224017175235}
Rank number:  1 {'model': 'ResNet', 'augmentation': 'rotation', 'mean_acc': 0.11666666666666667, 'std_acc': 0.07071067811865475, 'mean_f1': 0.06727716727716729, 'std_f1': 0.0762881015697721, 'mean_recall': 0.15531135531135531, 'std_recall': 0.07822206883455578, 'mean_precision': 0.09055829228243022, 'std_precision': 0.11796723968563749, 'loss': 2.2699051002661386, 'accuracy': 0.2222222222222222, 'precision': 0.13600454674623472, 'recall': 0.19007936507936507, 'f1': 0.1317955033472274

In [ ]:
# tutaj też sprawdzamy jak się modele zachowywały, wizualizacje i tym podobne etc.

## Few-shot learning

In [ ]:
#przyklad jak wybrać podzbior do fewshot learningu, wystarczy na wczytanym datasecie zastosować get_subset

transformations = basic_transforms(augmentation_type=None, model_type='ResNet')

train_dataset = get_train_dataset(transform=transformations)
val_dataset, test_dataset = get_val_train_dataset(model_type='ResNet')

few_shot_train =  get_subset(train_dataset)
few_shot_val =  get_subset(val_dataset)
few_shot_test =  get_subset(test_dataset)